# SpectralWaste transformer screen (Colab, Run all)

Before you start: **Runtime -> Change runtime type -> A100 GPU -> Save**. Then **Runtime -> Run all**. Keep this tab open until the last cell finishes.

What it does, in order:
1. Checks the GPU.
2. Downloads only the labeled data from the Hugging Face mirror (about 25 GB, 8 to 10 minutes) onto Colab's local disk. Drive is not used.
3. Clones Scott's `wp6-colab-gates` branch.
4. Restores our epoch-13 checkpoint. You upload two files from `~/sw_runs_backup/screen/` (`last.pt` and `metrics.jsonl`), so training resumes at epoch 14. Set `RESUME = False` in the second cell to start from epoch 1 instead (about 55 minutes).
5. Trains with `--standardize --lr 3e-4 --no-cosine` at half resolution, 30 epochs.
6. Zips the results and downloads `tf_results.zip` to your computer.

Colab's local disk disappears when the session ends, so wait for the download in the last cell.

In [ ]:
RESUME = True   # True: continue from the epoch-13 checkpoint you upload. False: start at epoch 1.
EPOCHS = 30
NUM_WORKERS = 4

import subprocess
print(subprocess.run("nvidia-smi --query-gpu=name,memory.total --format=csv,noheader",
                     shell=True, capture_output=True, text=True).stdout or "NO GPU: set the runtime type to A100")

In [ ]:
# Download only the labeled shards (train 16, validation 5, test 6) and keep the cubes and masks.
import os, subprocess, concurrent.futures as cf, tarfile, pathlib
BASE = "https://huggingface.co/datasets/ferpb/spectralwaste-segmentation/resolve/main/data"
ROOT = pathlib.Path("/content/sw_data"); TMP = ROOT / "_tar"; TMP.mkdir(parents=True, exist_ok=True)
MAP = {"train": "train", "validation": "val", "test": "test"}
SHARDS = [("train", f"train-{i:06d}") for i in range(16)] + \
         [("validation", f"validation-{i:06d}") for i in range(5)] + \
         [("test", f"test-{i:06d}") for i in range(6)]

def fetch(item):
    src, shard = item; split = MAP[src]; tp = TMP / f"{shard}.tar"
    subprocess.run(["curl", "-s", "-L", "--fail", "--retry", "5", "-C", "-", "-o", str(tp), f"{BASE}/{shard}.tar"], check=True)
    (ROOT / "hyper" / split).mkdir(parents=True, exist_ok=True)
    (ROOT / "labels_hyper_lt" / split).mkdir(parents=True, exist_ok=True)
    with tarfile.open(tp) as tf:
        for m in tf.getmembers():
            b = os.path.basename(m.name)
            if b.endswith(".hsi.tiff"):
                dst = ROOT / "hyper" / split / (b[:-len(".hsi.tiff")] + ".tiff")
            elif b.endswith(".label_hsi_lt.png"):
                dst = ROOT / "labels_hyper_lt" / split / (b[:-len(".label_hsi_lt.png")] + ".png")
            else:
                continue
            with tf.extractfile(m) as s, open(dst, "wb") as o:
                o.write(s.read())
    tp.unlink()
    return shard

with cf.ThreadPoolExecutor(4) as ex:
    for f in cf.as_completed([ex.submit(fetch, s) for s in SHARDS]):
        print("done", f.result())

counts = {sp: len(list((ROOT / "hyper" / sp).glob("*.tiff"))) for sp in ["train", "val", "test"]}
print(counts)
assert counts == {"train": 514, "val": 167, "test": 171}, "unexpected split sizes, stop and check the download"
print("data OK: 514 / 167 / 171")

In [ ]:
!git clone -q -b wp6-colab-gates https://github.com/scottn66/255-final-project.git /content/255-final-project
!pip install -q tifffile
!cd /content/255-final-project && git log --oneline -1

In [ ]:
#@title Upload the epoch-13 checkpoint (only if RESUME is True)
import os, shutil
OUT = "/content/sw_runs/tf"; os.makedirs(OUT, exist_ok=True)
if RESUME:
    print("Choose last.pt and metrics.jsonl from ~/sw_runs_backup/screen/")
    from google.colab import files
    up = files.upload()
    for name in up:
        shutil.move(name, f"{OUT}/{name}")
    assert os.path.exists(f"{OUT}/last.pt") and os.path.exists(f"{OUT}/metrics.jsonl"), "upload last.pt AND metrics.jsonl, then run this cell again"
    import torch
    c = torch.load(f"{OUT}/last.pt", map_location="cpu", weights_only=False)
    print("resuming from epoch", c["epoch"], " best val FG mIoU so far", round(c["best_val_fg_miou"], 4))
else:
    print("RESUME is False: training starts at epoch 1")


In [ ]:
resume_arg = "--resume /content/sw_runs/tf/last.pt" if RESUME else ""
!cd /content/255-final-project && PYTHONPATH=src SW_ROOT=/content/sw_data python -u -m sw.train \
    --model transformer --downsample 2 --epochs {EPOCHS} --batch-size 4 --num-workers {NUM_WORKERS} --device cuda \
    --no-cosine --lr 3e-4 --standardize {resume_arg} --out-dir /content/sw_runs/tf

In [ ]:
import json
rows = [json.loads(l) for l in open("/content/sw_runs/tf/metrics.jsonl")]
print("epoch  val_fg_mIoU")
for r in rows:
    print(f"{r['epoch']:>5}  {r['val_fg_miou']:.4f}")
best = max(rows, key=lambda r: r["val_fg_miou"])
print("\nbest val FG mIoU", round(best["val_fg_miou"], 4), "at epoch", best["epoch"])
print({k: round(v, 3) for k, v in best["per_class_iou"].items()})

# Zip the run folder and download it. Do this before the session ends.
from google.colab import files
!cd /content/sw_runs && zip -qr tf_results.zip tf
files.download("/content/sw_runs/tf_results.zip")